# Avaliação Multi-Sessão

Este notebook avalia sessões de agentes usando Strands Evals, um framework extensível de avaliação baseado em LLM que utiliza LLMs como juízes. Para cada sessão, ele busca traces do AgentCore Observability, executa avaliadores e registra os resultados de volta com os IDs de trace originais para correlação no painel.

**Este notebook demonstra dois avaliadores:**
- **OutputEvaluator**: Pontua a qualidade da resposta (relevância, precisão, completude)
- **TrajectoryEvaluator**: Pontua o uso de ferramentas (seleção, eficiência, sequência)

Strands Evals suporta avaliadores personalizados para praticamente qualquer tipo de avaliação. O poder do framework está no sistema de rubricas — defina seus critérios e o LLM os aplica de forma consistente.

**Fluxo de trabalho:**
1. Carregue sessões do notebook de descoberta (ou forneça IDs de sessão personalizados)
2. Para cada sessão: busque traces, crie casos de avaliação, execute avaliadores
3. Registre resultados no AgentCore em formato EMF
4. Gere estatísticas resumidas

**Pré-requisito:** Execute o notebook de descoberta de sessões primeiro, ou prepare uma lista de IDs de sessão.

## Onde Isto se Encaixa

Este é o **Notebook 2 (Opção A)** - avalie sessões usando rubricas personalizadas que você define.

![Notebook Workflow](images/notebook_workflow.svg)

## Como os Dados Fluem

O pipeline de avaliação transforma traces do AgentCore Observability em resultados pontuados:

![Evaluation Pipeline](images/evaluation_pipeline.svg)

## Configuração Inicial

Importe os módulos necessários, incluindo avaliadores do Strands Evals e classes utilitárias para interação com o AgentCore Observability. A configuração é carregada de `config.py`.

In [ ]:
import logging
import sys
from datetime import datetime, timedelta, timezone
from typing import List

sys.path.insert(0, ".")

from config import (
    AWS_REGION,
    AWS_ACCOUNT_ID,
    SOURCE_LOG_GROUP,
    EVAL_RESULTS_LOG_GROUP,
    LOOKBACK_HOURS,
    MAX_CASES_PER_SESSION,
    DISCOVERED_SESSIONS_PATH,
    RESULTS_JSON_PATH,
    EVALUATION_CONFIG_ID,
    setup_cloudwatch_environment,
)

from utils import (
    CloudWatchSessionMapper,
    ObservabilityClient,
    SessionDiscoveryResult,
    SessionInfo,
    send_evaluation_to_cloudwatch,
)

from strands_evals import Case, Experiment
from strands_evals.evaluators import OutputEvaluator, TrajectoryEvaluator
from strands_evals.types.trace import AgentInvocationSpan, ToolExecutionSpan

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

## Configuração

Defina nomes de avaliadores para métricas do CloudWatch. Esses nomes aparecem no painel do AgentCore Observability e devem seguir a convenção `Custom.NomeDoSeuAvaliador`. O `EVALUATION_CONFIG_ID` é carregado de `config.py`.

In [ ]:
# Custom evaluator names for CloudWatch metrics (customize for your use case)
OUTPUT_EVALUATOR_NAME = "Custom.OutputEvaluator"
TRAJECTORY_EVALUATOR_NAME = "Custom.TrajectoryEvaluator"

## Ambiente CloudWatch

Configure as variáveis de ambiente necessárias para registrar resultados de avaliação. Usa `SERVICE_NAME` de `config.py` para atributos de recurso OTEL.

In [ ]:
setup_cloudwatch_environment()

## Carregar Sessões

Carregue sessões a partir da saída JSON do notebook de descoberta. Alternativamente, defina `USE_JSON_FILE = False` e forneça IDs de sessão personalizados diretamente para reavaliação direcionada de sessões específicas.

In [ ]:
# Set to False to provide custom session IDs instead
USE_JSON_FILE = True

if USE_JSON_FILE:
    discovery_result = SessionDiscoveryResult.load_from_json(DISCOVERED_SESSIONS_PATH)
    sessions_to_process = discovery_result.sessions
else:
    # Provide custom session IDs here
    session_ids = [
        "your-session-id-here",
    ]
    sessions_to_process = [
        SessionInfo(
            session_id=sid,
            span_count=0,
            first_seen=datetime.now(timezone.utc),
            last_seen=datetime.now(timezone.utc),
            discovery_method="user_provided",
        )
        for sid in session_ids
    ]

print(f"Loaded {len(sessions_to_process)} sessions")

## Rubricas dos Avaliadores

As rubricas definem seus critérios de avaliação. O avaliador envia a rubrica junto com a saída do agente para um LLM, que atua como juiz e retorna uma pontuação (0.0-1.0) com uma explicação.

**Escrevendo rubricas eficazes:**
- Seja específico sobre o que constitui boa vs má qualidade
- Inclua âncoras de pontuação (o que significa 1.0 vs 0.5 vs 0.0?)
- Foque em critérios mensuráveis relevantes para o domínio do seu agente

Personalize estas rubricas abaixo. As rubricas padrão avaliam a qualidade geral da resposta e padrões de uso de ferramentas.

In [ ]:
output_rubric = """
Evaluate the agent's response based on:
1. Relevance: Does the response directly address the user's question?
2. Accuracy: Is the information factually correct?
3. Completeness: Does the response provide sufficient detail?

Score 0.0-1.0: 1.0=excellent, 0.5=adequate, 0.0=poor
"""

trajectory_rubric = """
Evaluate the agent's tool usage based on:
1. Tool Selection: Did the agent choose appropriate tools?
2. Efficiency: Were tools used without unnecessary calls?
3. Logical Sequence: Were tools used in a logical order?

Score 0.0-1.0: 1.0=optimal, 0.5=acceptable, 0.0=poor
"""

## Funções Auxiliares

Estas funções fazem a ponte entre traces do AgentCore Observability e Strands Evals:

- `task_fn(case)`: Retorna a resposta real do agente para o OutputEvaluator pontuar contra a rubrica.

- `trajectory_task_fn(case)`: Retorna tanto a resposta quanto a sequência de ferramentas para o TrajectoryEvaluator avaliar padrões de uso de ferramentas.

- `create_cases_from_session(session)`: Converte uma Session do Strands Eval em Cases de avaliação. Extrai prompts do usuário de AgentInvocationSpan, nomes de ferramentas de objetos ToolExecutionSpan e preserva o trace_id original para correlação no CloudWatch.

- `log_case_result_to_cloudwatch(case, ...)`: Envia resultados de avaliação para o AgentCore Observability usando o trace_id original, permitindo que você veja pontuações ao lado dos traces originais no painel.

In [ ]:
def task_fn(case: Case) -> str:
    """Return actual output from trace metadata."""
    return (case.metadata.get("actual_output", ""))


def trajectory_task_fn(case: Case):
    """Return output and trajectory from trace metadata."""
    return {"output": case.metadata.get("actual_output", ""), "trajectory": case.metadata.get("trajectory_for_eval", [])}

def log_case_result_to_cloudwatch(case: Case, evaluator_name: str, score: float, explanation: str, label: str = None) -> bool:
    """Log evaluation result to CloudWatch with original trace ID."""
    trace_id = case.metadata.get("trace_id", "")
    if not trace_id:
        return False
    return send_evaluation_to_cloudwatch(
        trace_id=trace_id,
        session_id=case.session_id,
        evaluator_name=evaluator_name,
        score=score,
        explanation=explanation,
        label=label,
        config_id=EVALUATION_CONFIG_ID,
    )


def create_cases_from_session(session, session_id: str, max_cases: int = None) -> List[Case]:
    """Create evaluation cases from a Strands Eval Session."""
    cases = []
    for i, trace in enumerate(session.traces):
        if max_cases and len(cases) >= max_cases:
            break
        agent_span = None
        tool_names = []
        for span in trace.spans:
            if isinstance(span, AgentInvocationSpan):
                agent_span = span
            elif isinstance(span, ToolExecutionSpan):
                tool_names.append(span.tool_call.name)
        if agent_span:
            case = Case(
                name=f"trace_{i+1}_{trace.trace_id[:8]}",
                input=agent_span.user_prompt or "",
                expected_output="",
                session_id=session_id,
                metadata={
                    "actual_output": agent_span.agent_response or "",
                    "actual_trajectory": tool_names,
                    "trace_id": trace.trace_id,
                    "tool_count": len(tool_names),
                },
            )
            cases.append(case)
    return cases

## Inicializar Cliente

Crie o `ObservabilityClient` para buscar traces e o `CloudWatchSessionMapper` para convertê-los.

O mapper transforma spans brutos do AgentCore Observability em objetos estruturados do Strands Eval:
- Agrupa spans por trace_id para reconstruir cada interação
- Extrai chamadas de ferramentas e as associa com seus resultados
- Identifica prompts do usuário (primeira mensagem) e respostas do agente (saída final)
- Produz AgentInvocationSpan (interação completa) e ToolExecutionSpan (cada uso de ferramenta)

In [ ]:
obs_client = ObservabilityClient(
    region_name=AWS_REGION,
    log_group=SOURCE_LOG_GROUP,
)
mapper = CloudWatchSessionMapper()

end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(hours=LOOKBACK_HOURS)
start_time_ms = int(start_time.timestamp() * 1000)
end_time_ms = int(end_time.timestamp() * 1000)

## Processar Sessões

O loop principal de avaliação. Para cada sessão:
1. Busque spans do AgentCore Observability
2. Converta spans para o formato Session do Strands Eval usando o mapper
3. Crie Cases de avaliação a partir de cada trace na sessão
4. Execute o OutputEvaluator em todos os cases
5. Execute o TrajectoryEvaluator nos cases que usaram ferramentas
6. Registre todos os resultados no AgentCore Observability com os IDs de trace originais para correlação no painel

O progresso é impresso para cada sessão. Erros são capturados e registrados sem interromper o loop.

In [ ]:
all_session_results = []
total_cases_evaluated = 0
total_logs_sent = 0
all_tools_used = set()

for session_idx, session_info in enumerate(sessions_to_process):
    session_id = session_info.session_id
    print(f"[{session_idx + 1}/{len(sessions_to_process)}] {session_id}")

    try:
        trace_data = obs_client.get_session_data(
            session_id=session_id,
            start_time_ms=start_time_ms,
            end_time_ms=end_time_ms,
            include_runtime_logs=False,
        )

        if not trace_data.spans:
            all_session_results.append({"session_id": session_id, "status": "skipped", "reason": "no_spans"})
            continue

        session = trace_data.to_session(mapper)
        cases = create_cases_from_session(session, session_id, MAX_CASES_PER_SESSION)

        if not cases:
            all_session_results.append({"session_id": session_id, "status": "skipped", "reason": "no_cases"})
            continue

        for case in cases:
            for tool in case.metadata.get("actual_trajectory", []):
                all_tools_used.add(tool)

        # Run Output Evaluator
        output_experiment = Experiment(cases=cases, evaluators=[OutputEvaluator(rubric=output_rubric)])
        output_results = output_experiment.run_evaluations(task_fn)
        output_report = output_results[0]

        output_logged = 0
        for i, case in enumerate(cases):
            if log_case_result_to_cloudwatch(case, OUTPUT_EVALUATOR_NAME, output_report.scores[i], output_report.reasons[i] if i < len(output_report.reasons) else ""):
                output_logged += 1

        # Run Trajectory Evaluator
        trajectory_cases = [c for c in cases if c.metadata.get("actual_trajectory")]
        trajectory_score = None
        trajectory_logged = 0

        if trajectory_cases:
            traj_eval_cases = [
                Case(name=c.name, input=c.input, expected_output=c.expected_output, session_id=c.session_id,
                     metadata={**c.metadata, "trajectory_for_eval": c.metadata.get("actual_trajectory", [])})
                for c in trajectory_cases
            ]
            trajectory_experiment = Experiment(
                cases=traj_eval_cases,
                evaluators=[TrajectoryEvaluator(rubric=trajectory_rubric, trajectory_description={"available_tools": list(all_tools_used)})]
            )
            trajectory_results = trajectory_experiment.run_evaluations(trajectory_task_fn)
            trajectory_report = trajectory_results[0]
            trajectory_score = trajectory_report.overall_score

            for i, case in enumerate(traj_eval_cases):
                if log_case_result_to_cloudwatch(case, TRAJECTORY_EVALUATOR_NAME, trajectory_report.scores[i], trajectory_report.reasons[i] if i < len(trajectory_report.reasons) else ""):
                    trajectory_logged += 1

        all_session_results.append({
            "session_id": session_id,
            "status": "completed",
            "case_count": len(cases),
            "output_score": output_report.overall_score,
            "trajectory_score": trajectory_score,
            "logs_sent": output_logged + trajectory_logged,
        })
        total_cases_evaluated += len(cases)
        total_logs_sent += output_logged + trajectory_logged

    except Exception as e:
        all_session_results.append({"session_id": session_id, "status": "error", "error": str(e)})

print(f"\nCompleted: {len([r for r in all_session_results if r['status'] == 'completed'])} sessions, {total_cases_evaluated} cases, {total_logs_sent} logs sent")

## Resumo

Estatísticas agregadas de todas as sessões avaliadas, incluindo taxa de conclusão, total de cases avaliados e pontuações médias para os avaliadores de saída e trajetória.

In [ ]:
completed = [r for r in all_session_results if r.get("status") == "completed"]
output_scores = [r["output_score"] for r in completed if r.get("output_score") is not None]
trajectory_scores = [r["trajectory_score"] for r in completed if r.get("trajectory_score") is not None]

print(f"Sessions: {len(completed)}/{len(all_session_results)} completed")
print(f"Cases evaluated: {total_cases_evaluated}")
print(f"CloudWatch logs sent: {total_logs_sent}")

if output_scores:
    print(f"Output score: avg={sum(output_scores)/len(output_scores):.2f}, min={min(output_scores):.2f}, max={max(output_scores):.2f}")
if trajectory_scores:
    print(f"Trajectory score: avg={sum(trajectory_scores)/len(trajectory_scores):.2f}, min={min(trajectory_scores):.2f}, max={max(trajectory_scores):.2f}")

## Resultados por Sessão

Resultados individuais para cada sessão mostrando pontuações de saída e trajetória. Sessões marcadas como "skipped" não tinham spans ou cases válidos. Sessões marcadas como "error" encontraram exceções durante o processamento.

In [ ]:
for i, r in enumerate(all_session_results):
    status = r.get("status", "unknown")
    if status == "completed":
        print(f"{i+1}. {r['session_id'][:20]}... output={r.get('output_score', 0):.2f} traj={r.get('trajectory_score') or '-'}")
    else:
        print(f"{i+1}. {r['session_id'][:20]}... {status}")

## Exportar Resultados

Salve os resultados de avaliação em JSON para análise adicional ou relatórios. A exportação inclui configuração, estatísticas resumidas e resultados por sessão.

In [ ]:
import json

export_data = {
    "evaluation_time": datetime.now(timezone.utc).isoformat(),
    "config": {
        "source_log_group": SOURCE_LOG_GROUP,
        "eval_results_log_group": EVAL_RESULTS_LOG_GROUP,
        "output_evaluator": OUTPUT_EVALUATOR_NAME,
        "trajectory_evaluator": TRAJECTORY_EVALUATOR_NAME,
    },
    "summary": {
        "total_sessions": len(all_session_results),
        "completed_sessions": len(completed),
        "total_cases": total_cases_evaluated,
        "avg_output_score": sum(output_scores) / len(output_scores) if output_scores else None,
        "avg_trajectory_score": sum(trajectory_scores) / len(trajectory_scores) if trajectory_scores else None,
    },
    "session_results": all_session_results,
}

with open(RESULTS_JSON_PATH, "w") as f:
    json.dump(export_data, f, indent=2)

print(f"Exported to {RESULTS_JSON_PATH}")